# 문헌 정보 추출 실습

**Literature Mining · Text Mining · 문헌 마이닝**

논문과 특허 문장에서 조성·공정·물성 값을 자동으로 뽑아 데이터로 만드는 작업. 추출값은 원문 대조가 필요하다.

소재 분야에서 이해하기: 합성 온도 표현을 표준 단위로 정리해 데이터셋을 만든다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Google ML 용어집](https://developers.google.com/machine-learning/glossary)

## 1. 규칙 기반 추출부터

문장에서 합성 온도와 물성값을 뽑아 표로 만듭니다. 단위 정규화까지 합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

import re

abstracts = [
 "The alloy was sintered at 780 C for 4 h, giving a Vickers hardness of 431 HV.",
 "Samples annealed at 1050 K for 30 min showed a hardness of 4.2 GPa.",
 "Sintering at 800 degrees C yielded a hardness of 455 HV after 2 hours.",
 "We report a band gap of 1.45 eV for the film deposited at 300 C.",
 "The catalyst was calcined at 550 C; activity increased by 30%.",
]
for text in abstracts:
    print('-', text)

In [ ]:
TEMPERATURE = re.compile(r'(\d+(?:\.\d+)?)\s*(?:degrees\s*)?(C|K)\b')
HARDNESS = re.compile(r'hardness of\s*(\d+(?:\.\d+)?)\s*(HV|GPa)')
DURATION = re.compile(r'for\s*(\d+(?:\.\d+)?)\s*(h|hours?|min)')

def to_celsius(value, unit):
    return value - 273.15 if unit == 'K' else value

def to_hv(value, unit):
    return value * 1000.0 / 9.807 if unit == 'GPa' else value       # 근사 변환

def to_hours(value, unit):
    return value / 60.0 if unit.startswith('min') else value

rows = []
for text in abstracts:
    temperature = TEMPERATURE.search(text)
    hardness = HARDNESS.search(text)
    duration = DURATION.search(text)
    rows.append({
        'temperature_C': round(to_celsius(float(temperature.group(1)), temperature.group(2)), 1) if temperature else None,
        'hold_h': round(to_hours(float(duration.group(1)), duration.group(2)), 2) if duration else None,
        'hardness_HV': round(to_hv(float(hardness.group(1)), hardness.group(2)), 1) if hardness else None,
        'source': text[:45] + '...',
    })

import pandas as pd
table = pd.DataFrame(rows)
print(table.to_string())

## 2. 추출 결과를 검증합니다

추출은 반드시 원문 대조와 범위 검사를 통과해야 데이터로 쓸 수 있습니다.

In [ ]:
RANGES = {'temperature_C': (0, 3000), 'hold_h': (0.01, 200), 'hardness_HV': (10, 2000)}

def validate(row):
    problems = []
    for field, (low, high) in RANGES.items():
        value = row[field]
        if value is None:
            problems.append(field + ' 누락')
        elif not low <= value <= high:
            problems.append('%s 범위 밖(%.1f)' % (field, value))
    return problems

usable = 0
for index, row in table.iterrows():
    problems = validate(row)
    status = '사용 가능' if not problems else ' / '.join(problems)
    usable += not problems
    print('%-48s %s' % (row['source'][:46], status))
print('\n%d개 문장 중 %d개만 데이터로 사용 가능' % (len(table), usable))
print('규칙 기반 추출은 표현이 조금만 달라져도 놓칩니다("annealed" 와 "sintered", "degrees C" 등).')
print('LLM 을 쓰면 표현 변화에 강해지지만, 없는 값을 만들어낼 수 있어 원문 대조 검증이 더 중요해집니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#literature-mining)을 여세요.